# H7: Individual-difference story exploration

Goals:
1. **Verify the §2.6 draft claims** on the current ω/κ model in BOTH samples (the published numbers come from old analyses on a different model).
   - mean anxiety → DASS-Anxiety (controlling ω, κ)
   - mean confidence → DASS-Depression (controlling ω, κ)
   - mean confidence → AMI (controlling ω, κ)
   - anxiety-tracking → AMI (controlling ω, κ)
   - AMI → foraging outcomes (escape, earnings, choice-shift, pct_opt)

2. **Quadrant profiles** in (ω, κ) plane: behavior + affect + clinical contrasts.

3. **Model–self discrepancy** as a candidate individual-difference dimension: residualised confidence (or escape) → clinical.

Each test reported per sample so we can see what survives independent replication.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys
if '/workspace/notebooks/analysis' not in sys.path:
    sys.path.insert(0, '/workspace/notebooks/analysis')
import numpy as np, pandas as pd
import bambi as bmb, arviz as az
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import pearsonr, zscore, f_oneway
from config import *
from load_data import load_both

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9,
                     'axes.spines.right': False, 'axes.spines.top': False})

STATS_DIR = REPO_ROOT / 'results' / 'stats' / 'individual_diffs'
STATS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = REPO_ROOT / 'results' / 'figs' / 'individual_diffs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

exp_data, conf_data = load_both()
all_data = {'exploratory': exp_data, 'confirmatory': conf_data}
all_data = {k: v for k, v in all_data.items() if v is not None}

for name, d in all_data.items():
    m = d['master']
    print(f'{d["config"].label}: master={len(m)} subjects')
    for col in ['mean_anxiety','mean_confidence','anx_slope','anx_calibration',
                'AMI_Total','DASS21_Anxiety','DASS21_Depression']:
        if col in m.columns:
            print(f'  {col}: {m[col].notna().sum()} non-null')

Exploratory (N=290): stage5_filtered_data_20260403_133425 | params: 290 subjects
Confirmatory (N=281): stage5_filtered_data_20260403_142413 | params: 281 subjects
GPU: No


Exploratory (N=290): master=290 subjects
  mean_anxiety: 290 non-null
  mean_confidence: 290 non-null
  anx_slope: 290 non-null
  anx_calibration: 286 non-null
  AMI_Total: 290 non-null
  DASS21_Anxiety: 290 non-null
  DASS21_Depression: 290 non-null
Confirmatory (N=281): master=281 subjects
  mean_anxiety: 281 non-null
  mean_confidence: 281 non-null
  anx_slope: 281 non-null
  anx_calibration: 277 non-null
  AMI_Total: 281 non-null
  DASS21_Anxiety: 281 non-null
  DASS21_Depression: 281 non-null


## Helper: partial Bayesian regression with z-scoring

Standardize DV and predictor to make β comparable across scales.

In [2]:
def fit_partial(master, dv, predictor, controls=('omega_z','kappa_z')):
    cols = [dv, predictor] + list(controls)
    df = master[cols].dropna().copy()
    if len(df) < 30:
        return None
    for c in [dv, predictor]:
        v = df[c].astype(float)
        df[c] = (v - v.mean()) / v.std() if v.std() > 0 else 0
    rhs = predictor + (' + ' + ' + '.join(controls) if controls else '')
    mod = bmb.Model(f'{dv} ~ {rhs}', data=df)
    res = mod.fit(**BKW)
    s = az.summary(res, hdi_prob=0.95)
    lo, hi = s.loc[predictor,'hdi_2.5%'], s.loc[predictor,'hdi_97.5%']
    return {'n': len(df), 'b': s.loc[predictor,'mean'],
            'lo': lo, 'hi': hi, 'sig': (lo > 0) or (hi < 0)}

def report(claims_results, header):
    print(f'\n{"="*70}\n{header}\n{"="*70}')
    for label, results in claims_results:
        print(f'\n{label}')
        for name, r in results.items():
            if r is None:
                print(f'  {name}: --')
                continue
            flag = '★' if r['sig'] else ' '
            print(f'  {name:<13} N={r["n"]:>3}  β={r["b"]:+.3f}  [{r["lo"]:+.3f}, {r["hi"]:+.3f}]  {flag}')

## Step 1 — Verify §2.6 task-affect → clinical claims (controlling ω, κ)

In [3]:
claims = [
    ('mean_anxiety → DASS21_Anxiety',    'DASS21_Anxiety',    'mean_anxiety',    '+'),
    ('mean_anxiety → DASS21_Depression', 'DASS21_Depression', 'mean_anxiety',    '+/0'),
    ('mean_confidence → DASS21_Depression','DASS21_Depression','mean_confidence', '−'),
    ('mean_confidence → AMI_Total',      'AMI_Total',         'mean_confidence', '−'),
    ('anx_calibration → AMI_Total',      'AMI_Total',         'anx_calibration', '+'),
    ('anx_slope → AMI_Total',            'AMI_Total',         'anx_slope',       '+'),
]

step1_results = []
for label, clinical, affect, expected in claims:
    res = {}
    for name, d in all_data.items():
        res[name] = fit_partial(d['master'], clinical, affect)
    step1_results.append((f'{label} (expected {expected})', res))

report(step1_results, 'STEP 1 — task affect → clinical (controlling ω, κ)')

# Save
rows = []
for label, res in step1_results:
    for name, r in res.items():
        if r is None: continue
        rows.append({'test': label, 'sample': name, **r})
pd.DataFrame(rows).to_csv(STATS_DIR / 'step1_affect_to_clinical.csv', index=False)

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, mean_anxiety, omega_z, kappa_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


Initializing NUTS using jitter+adapt_diag...


...

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, anx_slope, omega_z, kappa_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.



STEP 1 — task affect → clinical (controlling ω, κ)

mean_anxiety → DASS21_Anxiety (expected +)
  exploratory   N=290  β=+0.252  [+0.137, +0.363]  ★
  confirmatory  N=281  β=+0.195  [+0.080, +0.310]  ★

mean_anxiety → DASS21_Depression (expected +/0)
  exploratory   N=290  β=+0.213  [+0.094, +0.322]  ★
  confirmatory  N=281  β=+0.115  [-0.003, +0.235]   

mean_confidence → DASS21_Depression (expected −)
  exploratory   N=290  β=-0.202  [-0.323, -0.088]  ★
  confirmatory  N=281  β=-0.068  [-0.189, +0.049]   

mean_confidence → AMI_Total (expected −)
  exploratory   N=290  β=-0.260  [-0.374, -0.146]  ★
  confirmatory  N=281  β=-0.129  [-0.243, -0.006]  ★

anx_calibration → AMI_Total (expected +)
  exploratory   N=286  β=+0.064  [-0.055, +0.178]   
  confirmatory  N=277  β=+0.154  [+0.041, +0.271]  ★

anx_slope → AMI_Total (expected +)
  exploratory   N=290  β=+0.016  [-0.101, +0.131]   
  confirmatory  N=281  β=+0.153  [+0.035, +0.267]  ★


## Step 2 — AMI → foraging outcomes (univariate and partialling ω, κ)

In [4]:
outcomes = ['escape_rate','earnings','choice_shift','pct_opt']

step2_uni = []
step2_partial = []
for outcome in outcomes:
    res_u = {n: fit_partial(d['master'], outcome, 'AMI_Total', controls=()) for n, d in all_data.items()}
    res_p = {n: fit_partial(d['master'], outcome, 'AMI_Total') for n, d in all_data.items()}
    step2_uni.append((f'{outcome} ~ AMI_Total', res_u))
    step2_partial.append((f'{outcome} ~ AMI_Total + ω + κ', res_p))

report(step2_uni, 'STEP 2a — AMI → outcomes (univariate)')
report(step2_partial, 'STEP 2b — AMI → outcomes (controlling ω, κ)')

rows = []
for label, res in step2_uni + step2_partial:
    for name, r in res.items():
        if r is None: continue
        rows.append({'test': label, 'sample': name, **r})
pd.DataFrame(rows).to_csv(STATS_DIR / 'step2_ami_to_outcomes.csv', index=False)

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, AMI_Total]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


Initializing NUTS using jitter+adapt_diag...


...

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, AMI_Total, omega_z, kappa_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.



STEP 2a — AMI → outcomes (univariate)

escape_rate ~ AMI_Total
  exploratory   N=290  β=+0.308  [+0.198, +0.414]  ★
  confirmatory  N=281  β=+0.424  [+0.320, +0.528]  ★

earnings ~ AMI_Total
  exploratory   N=290  β=+0.291  [+0.186, +0.407]  ★
  confirmatory  N=281  β=+0.387  [+0.283, +0.495]  ★

choice_shift ~ AMI_Total
  exploratory   N=290  β=+0.121  [+0.011, +0.238]  ★
  confirmatory  N=281  β=+0.234  [+0.115, +0.344]  ★

pct_opt ~ AMI_Total
  exploratory   N=290  β=+0.050  [-0.066, +0.164]   
  confirmatory  N=281  β=-0.035  [-0.155, +0.081]   

STEP 2b — AMI → outcomes (controlling ω, κ)

escape_rate ~ AMI_Total + ω + κ
  exploratory   N=290  β=+0.294  [+0.182, +0.400]  ★
  confirmatory  N=281  β=+0.404  [+0.304, +0.514]  ★

earnings ~ AMI_Total + ω + κ
  exploratory   N=290  β=+0.288  [+0.174, +0.399]  ★
  confirmatory  N=281  β=+0.405  [+0.292, +0.516]  ★

choice_shift ~ AMI_Total + ω + κ
  exploratory   N=290  β=+0.128  [+0.014, +0.246]  ★
  confirmatory  N=281  β=+0.249  [+0

## Step 3 — Quadrant profiles in (ω, κ) plane

Median split on standardized ω and κ → HH, HL, LH, LL. Compare on outcomes, affect, and clinical scales.

In [5]:
profile_cols = ['escape_rate','earnings','pct_opt','mean_vigor','choice_shift',
                'mean_anxiety','mean_confidence','anx_calibration',
                'AMI_Total','PHQ9_Total','DASS21_Anxiety','DASS21_Depression','STAI_Trait']

all_profiles = {}
for name, d in all_data.items():
    m = d['master'].copy()
    m['ω_hi'] = np.where(m['omega_z'] > 0, 'H', 'L')
    m['κ_hi'] = np.where(m['kappa_z'] > 0, 'H', 'L')
    m['quad'] = m['ω_hi'] + m['κ_hi']
    quad_means = m.groupby('quad')[profile_cols].mean().round(3)
    quad_n = m['quad'].value_counts()
    quad_means = quad_means.reindex(['HH','HL','LH','LL'])
    quad_means['n'] = quad_n.reindex(['HH','HL','LH','LL']).values
    all_profiles[name] = quad_means
    print(f'\n=== {name} (N by quadrant: {dict(quad_n)}) ===')
    print(quad_means.to_string())

    # ANOVA for each col
    print('\nANOVA across quadrants (F, p):')
    for col in profile_cols:
        groups = [m[m['quad']==q][col].dropna().values for q in ['HH','HL','LH','LL']]
        if all(len(g) > 2 for g in groups):
            F, p = f_oneway(*groups)
            flag = '★' if p < 0.05 else ' '
            print(f'  {col:<22} F={F:.2f}, p={p:.4f} {flag}')

# Save
for name, qm in all_profiles.items():
    qm.to_csv(STATS_DIR / f'profiles_{name}.csv')


=== exploratory (N by quadrant: {'HH': np.int64(104), 'LL': np.int64(92), 'LH': np.int64(49), 'HL': np.int64(45)}) ===
      escape_rate  earnings  pct_opt  mean_vigor  choice_shift  mean_anxiety  mean_confidence  anx_calibration  AMI_Total  PHQ9_Total  DASS21_Anxiety  DASS21_Depression  STAI_Trait    n
quad                                                                                                                                                                                     
HH          0.422    -4.529    0.532       0.894         0.431         4.459            2.850            0.335     28.106       6.865           7.019             10.231      51.553  104
HL          0.446    24.956    0.627       1.255         0.566         4.415            2.836            0.310     30.978       8.822           8.711             13.556      50.844   45
LH          0.340    -5.918    0.667       0.789         0.512         4.472            3.408            0.281     25.878       7.490   

## Step 4 — Model–self discrepancy as a candidate individual-difference

Hypothesis: it's not your ω/κ that maps onto clinical, it's how *misaligned* your subjective state is with what your behavior says you should feel. Two operationalisations:

- **conf_resid**: residual of mean_confidence after regressing on escape_rate (overconfidence/underconfidence)
- **escape_resid**: residual of escape_rate after regressing on ω + κ (out-performing or under-performing what your parameters predict)

In [6]:
for name, d in all_data.items():
    m = d['master'].copy()

    # conf_resid: confidence above/below what your escape rate justifies
    df1 = m[['mean_confidence','escape_rate']].dropna()
    if len(df1) > 30:
        f1 = smf.ols('mean_confidence ~ escape_rate', data=df1).fit()
        m.loc[df1.index, 'conf_resid'] = f1.resid

    # escape_resid: escape rate above/below what your ω, κ predict
    df2 = m[['escape_rate','omega_z','kappa_z']].dropna()
    if len(df2) > 30:
        f2 = smf.ols('escape_rate ~ omega_z + kappa_z', data=df2).fit()
        m.loc[df2.index, 'escape_resid'] = f2.resid

    d['master'] = m

    print(f'\n=== {name} ===')
    for resid_col, expected in [('conf_resid','overconf people'), ('escape_resid','overperformers')]:
        if resid_col not in m.columns: continue
        print(f'\n{resid_col} ({expected})')
        for clinical in ['AMI_Total','PHQ9_Total','DASS21_Anxiety','DASS21_Depression','STAI_Trait','OASIS_Total']:
            if clinical not in m.columns: continue
            df = m[[resid_col, clinical]].dropna()
            if len(df) < 30: continue
            r, p = pearsonr(df[resid_col], df[clinical])
            flag = '★' if p < 0.05 else ' '
            print(f'  → {clinical:<22} r={r:+.3f}  p={p:.4f}  N={len(df)}  {flag}')


=== exploratory ===

conf_resid (overconf people)
  → AMI_Total              r=-0.248  p=0.0000  N=290  ★
  → PHQ9_Total             r=-0.153  p=0.0090  N=290  ★
  → DASS21_Anxiety         r=-0.109  p=0.0627  N=290   
  → DASS21_Depression      r=-0.176  p=0.0026  N=290  ★
  → STAI_Trait             r=+0.175  p=0.0028  N=289  ★
  → OASIS_Total            r=-0.154  p=0.0085  N=290  ★

escape_resid (overperformers)
  → AMI_Total              r=+0.301  p=0.0000  N=290  ★
  → PHQ9_Total             r=+0.070  p=0.2321  N=290   
  → DASS21_Anxiety         r=-0.090  p=0.1275  N=290   
  → DASS21_Depression      r=+0.069  p=0.2430  N=290   
  → STAI_Trait             r=-0.059  p=0.3159  N=289   
  → OASIS_Total            r=-0.039  p=0.5033  N=290   

=== confirmatory ===

conf_resid (overconf people)
  → AMI_Total              r=-0.170  p=0.0042  N=281  ★
  → PHQ9_Total             r=-0.050  p=0.3999  N=281   
  → DASS21_Anxiety         r=+0.091  p=0.1272  N=281   
  → DASS21_Depression     

## Step 5 — Anxiety-tracking as an individual difference (sanity)

anx_calibration = within-subject r(anxiety, threat). The H5 finding says this adds outcome prediction beyond ω, κ. Question: does it map onto clinical scales?

In [7]:
for name, d in all_data.items():
    m = d['master']
    if 'anx_calibration' not in m.columns: continue
    print(f'\n=== {name} ===')
    for clinical in ['AMI_Total','PHQ9_Total','DASS21_Anxiety','DASS21_Depression','STAI_Trait','OASIS_Total','STICSA_Total']:
        if clinical not in m.columns: continue
        df = m[['anx_calibration', clinical]].dropna()
        if len(df) < 30: continue
        r, p = pearsonr(df['anx_calibration'], df[clinical])
        flag = '★' if p < 0.05 else ' '
        print(f'  anx_calibration → {clinical:<20} r={r:+.3f}  p={p:.4f}  {flag}')


=== exploratory ===
  anx_calibration → AMI_Total            r=+0.070  p=0.2403   
  anx_calibration → PHQ9_Total           r=+0.007  p=0.9012   
  anx_calibration → DASS21_Anxiety       r=+0.012  p=0.8357   
  anx_calibration → DASS21_Depression    r=+0.056  p=0.3438   
  anx_calibration → STAI_Trait           r=-0.057  p=0.3389   
  anx_calibration → OASIS_Total          r=-0.012  p=0.8452   
  anx_calibration → STICSA_Total         r=+0.064  p=0.2800   

=== confirmatory ===
  anx_calibration → AMI_Total            r=+0.152  p=0.0112  ★
  anx_calibration → PHQ9_Total           r=+0.062  p=0.3014   
  anx_calibration → DASS21_Anxiety       r=-0.072  p=0.2331   
  anx_calibration → DASS21_Depression    r=+0.085  p=0.1560   
  anx_calibration → STAI_Trait           r=-0.103  p=0.0865   
  anx_calibration → OASIS_Total          r=+0.141  p=0.0190  ★
  anx_calibration → STICSA_Total         r=+0.005  p=0.9313   


## Verdict

In [8]:
print('=' * 70)
print('H7 — INDIVIDUAL DIFFERENCES VERDICT')
print('=' * 70)

def survivors(results):
    out = []
    for label, res in results:
        if all(r is not None and r['sig'] for r in res.values()):
            sig_n = sum(1 for r in res.values() if r['sig'])
            out.append((label, sig_n, len(res)))
    return out

def either(results):
    out = []
    for label, res in results:
        sigs = [n for n, r in res.items() if r is not None and r['sig']]
        if sigs:
            out.append((label, sigs))
    return out

print('\nSTEP 1 — task affect → clinical (controlling ω,κ)')
print('Replicates in BOTH samples:')
for label, _, _ in survivors(step1_results):
    print(f'  ★ {label}')
if not survivors(step1_results):
    print('  none')
print('Significant in at least one sample:')
for label, samples in either(step1_results):
    print(f'  {label} → {samples}')

print('\nSTEP 2 — AMI → outcomes (univariate)')
for label, _, _ in survivors(step2_uni):
    print(f'  ★ {label}')
if not survivors(step2_uni):
    print('  none replicate in both samples')
    for label, samples in either(step2_uni):
        print(f'  (single-sample only) {label} → {samples}')

print('\nSTEP 2 — AMI → outcomes (controlling ω, κ)')
for label, _, _ in survivors(step2_partial):
    print(f'  ★ {label}')
if not survivors(step2_partial):
    print('  none replicate in both samples')

print('\nFiles saved to:', STATS_DIR)

H7 — INDIVIDUAL DIFFERENCES VERDICT

STEP 1 — task affect → clinical (controlling ω,κ)
Replicates in BOTH samples:
  ★ mean_anxiety → DASS21_Anxiety (expected +)
  ★ mean_confidence → AMI_Total (expected −)
Significant in at least one sample:
  mean_anxiety → DASS21_Anxiety (expected +) → ['exploratory', 'confirmatory']
  mean_anxiety → DASS21_Depression (expected +/0) → ['exploratory']
  mean_confidence → DASS21_Depression (expected −) → ['exploratory']
  mean_confidence → AMI_Total (expected −) → ['exploratory', 'confirmatory']
  anx_calibration → AMI_Total (expected +) → ['confirmatory']
  anx_slope → AMI_Total (expected +) → ['confirmatory']

STEP 2 — AMI → outcomes (univariate)
  ★ escape_rate ~ AMI_Total
  ★ earnings ~ AMI_Total
  ★ choice_shift ~ AMI_Total

STEP 2 — AMI → outcomes (controlling ω, κ)
  ★ escape_rate ~ AMI_Total + ω + κ
  ★ earnings ~ AMI_Total + ω + κ
  ★ choice_shift ~ AMI_Total + ω + κ

Files saved to: /workspace/results/stats/individual_diffs
